# Machine Learning 1 - Nearest Neighbors and Decision Trees

## Lab objectives

* Classification with decision trees and random forests.
* Cross-validation and evaluation.

In [ ]:
from lab_tools import CIFAR10, get_hog_image
dataset = CIFAR10("C:/Users/regna/Documents/ULB/MA2/Q2/INFO_H501/INFO_H501/docs/LABS/CIFAR10/")


Pre-loading training data
Pre-loading test data


# 1. Nearest Neighbor

The following example uses the Nearest Neighbor algorithm on the Histogram of Gradient decriptors in the dataset.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

clf = KNeighborsClassifier(n_neighbors=1)
clf.fit( dataset.train['hog'], dataset.train['labels'] )

KNeighborsClassifier(n_neighbors=1)

* What is the **descriptive performance** of this classifier ?


In [9]:
from sklearn.metrics import accuracy_score, confusion_matrix

pred_train = clf.predict(dataset.train['hog'])
score_descr = accuracy_score(dataset.train['labels'],pred_train)
print(score_descr)
cm = confusion_matrix(dataset.train['labels'], pred_train)
print(cm)

0.9398
[[4590  258  152]
 [  51 4787  162]
 [  17  263 4720]]


* Modify the code to estimate the **predictive performance**.


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_val, Y_train, Y_val = train_test_split(dataset.train['hog'], 
    dataset.train["labels"],
    test_size=0.2,
    random_state=0,
    stratify=dataset.train["labels"]
)

clf =  KNeighborsClassifier(n_neighbors=1)
clf.fit(X_train,Y_train)
pred_val = clf.predict(X_val)

score_pred = accuracy_score(Y_val, pred_val)
cm = confusion_matrix(Y_val, pred_val)

print(f"Descriptive accuracy: {score_descr:.4f}")
print(f"Validation accuracy: {score_pred:.4f}")
print(cm)


Descriptive accuracy: 0.9398
Validation accuracy: 0.6990
[[590 258 152]
 [ 51 787 162]
 [ 17 263 720]]


* Use cross-validation to find the best hyper-parameters for this method.

In [11]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X_train_val = dataset.train["hog"]
Y_train_val = dataset.train["labels"]

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

param_grid = {
    "knn__n_neighbors": [1, 3, 5, 7, 9, 11, 15, 21],
    "knn__weights": ["uniform", "distance"], #importance des voisins
    "knn__p": [1, 2] #1: distance Manhattan, #2: distance euclidienne
}

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train_val, Y_train_val)

print("Best parameters:", grid.best_params_)
print("Best CV accuracy:", grid.best_score_)



Best parameters: {'knn__n_neighbors': 7, 'knn__p': 1, 'knn__weights': 'distance'}
Best CV accuracy: 0.7871333333333334


Validation sur test set

In [12]:
X_test= dataset.test['hog']
Y_test = dataset.test['labels']
pred_test = grid.predict(X_test) #garde automatiquement le meilleur 
score_pred = accuracy_score(Y_test, pred_test)
cm_pred= confusion_matrix(Y_test, pred_test)
print(f"Test accuracy: {score_pred:.4f}")
print("Test confusion matrix:")
print(cm_pred)

Test accuracy: 0.7920
Test confusion matrix:
[[766 169  65]
 [ 82 788 130]
 [ 18 160 822]]


## 2. Decision Trees

[Decision Trees](http://scikit-learn.org/stable/modules/tree.html#tree) classify the data by splitting the feature space according to simple, single-feature rules. Scikit-learn uses the [CART](https://en.wikipedia.org/wiki/Predictive_analytics#Classification_and_regression_trees_.28CART.29) algorithm for [its implementation](http://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) of the classifier. 

* **Create a simple Decision Tree classifier** using scikit-learn and train it on the HoG training set.
* Use cross-validation to find the best hyper-paramters for this method.

In [13]:
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix

X_train_val = dataset.train['hog']
Y_train_val = dataset.train['labels']

X_test = dataset.test['hog']
Y_test = dataset.test['labels']


param_grid = {
    'max_depth': [5,10,15,20,None],
    'min_samples_leaf': [1,2,5,10],
    'criterion': ['gini','entropy']
}

grid = GridSearchCV(
    DecisionTreeClassifier(),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train_val,Y_train_val)

print("Best parameters:", grid.best_params_)
print(f"Best cross-validation accuracy: {grid.best_score_:.4f}")

pred_test = grid.predict(X_test) #utilise automatiquement le meilleur modèle trouvé

score_pred = accuracy_score(Y_test, pred_test)
cm = confusion_matrix(Y_test, pred_test)

print(f"Test accuracy: {score_pred:.4f}")
print("Test set Confusion matrix:")
print(cm)


Best parameters: {'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 1}
Best cross-validation accuracy: 0.6000
Test accuracy: 0.6013
Test set Confusion matrix:
[[604 244 152]
 [160 621 219]
 [118 303 579]]


## 3. Random Forests

[Random Forest](http://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) classifiers use multiple decision trees trained on "weaker" datasets (less data and/or less features), averaging the results so as to reduce over-fitting.

* Use scikit-learn to **create a Random Forest classifier** on the CIFAR data. 
* Use cross-validation to find the best hyper-paramters for this method.

In [14]:
from sklearn import ensemble
from sklearn.ensemble import RandomForestClassifier
from sklearn import ensemble
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix

X_train_val = dataset.train["hog"]
Y_train_val = dataset.train["labels"]

X_test = dataset.test["hog"]
Y_test = dataset.test["labels"]

rf = RandomForestClassifier(
    n_estimators = 100, 
    random_state = 0
)

#Cross Validation (gridsearch)
param_grid = {
    "n_estimators": [50,100],
    "max_depth": [10,None],
    "min_samples_leaf": [1,5]
}

grid = GridSearchCV(rf,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train_val,Y_train_val)
print("Best parameters:", grid.best_params_)
print(f"Best cross-validation accuracy: {grid.best_score_:.4f}")

pred_test = grid.predict(X_test)

score_pred = accuracy_score(Y_test, pred_test)
cm = confusion_matrix(Y_test, pred_test)

print(f"Test accuracy: {score_pred:.4f}")
print("Confusion matrix:")
print(cm)



Best parameters: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 100}
Best cross-validation accuracy: 0.7577
Test accuracy: 0.7670
Confusion matrix:
[[785 164  51]
 [130 734 136]
 [ 55 163 782]]
